# Prédiction du prix des taxis à New York : analyse exploratoire et création de variables

Notebook de l'article [Analyse exploratoire du prix des taxis](https://moncoachdata.com/blog/prediction-du-prix-des-taxis-analyse/) (MonCoachData), version modernisée.

**Données** : [New York City Taxi Fare Prediction (Kaggle)](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/data). Télécharge `train.csv` et `test.csv` depuis l'onglet *Data* (compte Kaggle gratuit, et règlement de la compétition à accepter), puis place-les dans le même dossier que ce notebook. Dans Google Colab, la section 0 s'en charge.

**Ce qui a changé par rapport à la version d'origine**
- plus de dépendances inutiles (`folium`, `geopy`, `sklearn.preprocessing.Imputer`, qui n'existe plus) : seulement `numpy`, `pandas`, `matplotlib`, `seaborn` ;
- les variables sont créées par des fonctions **vectorisées** (colonnes entières) au lieu de `apply(axis=1)` ligne par ligne : quelques secondes au lieu de longues minutes sur 6 millions de lignes ;
- les mêmes fonctions servent pour le jeu d'entraînement et le jeu de test ;
- les nuages de points et les densités utilisent un **échantillon** : tracer 6 millions de points ne montre rien de plus ;
- les graphiques ont des légendes, et la carte des longs trajets est faite avec Matplotlib ;
- les noms des colonnes produites et des fichiers de sortie (`train_cleaned.csv`, `test_cleaned.csv`) sont identiques à l'ancienne version, pour la partie 2.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("fivethirtyeight")
pd.set_option("display.max_colwidth", None)

# Échantillon utilisé pour les graphiques lourds (densités, nuages de points)
TAILLE_ECHANTILLON = 500_000

def echantillon_de(donnees, n=TAILLE_ECHANTILLON, graine=0):
    # Échantillon aléatoire de n lignes (ou tout le jeu s'il est plus petit)
    return donnees.sample(min(n, len(donnees)), random_state=graine)


# Les graphiques sont aussi enregistrés en haute résolution dans le dossier images/
Path("images").mkdir(exist_ok=True)


def lire_csv(nom, **options):
    # Lit `nom` (ex. "train.csv") ou, à défaut, `nom` + ".zip" : Pandas lit directement dans l'archive
    chemin = Path(nom)
    if not chemin.exists():
        chemin = Path(nom + ".zip")
        if chemin.exists() and not zipfile.is_zipfile(chemin):
            raise ValueError(f"{chemin} n'est pas une archive zip valide (téléversement ou téléchargement "
                             "interrompu ?). Supprime ce fichier, puis relance la cellule de la section 0.")
    return pd.read_csv(chemin, **options)


def montrer(nom):
    plt.savefig(f"images/{nom}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 0. Récupérer les données (Google Colab uniquement)

Dans Google Colab, les fichiers de la compétition ne sont pas là. Cette cellule les télécharge depuis Kaggle ; en local, elle ne fait rien (place `train.csv` et `test.csv` à côté du notebook ; les versions `.zip` conviennent aussi, Pandas les lit directement).

Avant de l'exécuter :
1. accepte le règlement de la [compétition](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/rules) avec ton compte Kaggle ;
2. crée un jeton API (Kaggle > Settings > API > *Create New Token*) : tu obtiens soit une chaîne de caractères à coller dans la cellule, soit un fichier `kaggle.json` (dans ce cas, laisse le champ vide : la cellule te demandera de le téléverser).

Attention : `train.csv` pèse plusieurs Go (le téléchargement prend quelques minutes). On n'en lira ensuite que 6 millions de lignes.

In [ ]:
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def fichier_valide(nom):
    # Le CSV est là, ou bien une archive zip complète (un téléversement interrompu donne un zip invalide)
    archive = Path(nom + ".zip")
    return Path(nom).exists() or (archive.exists() and zipfile.is_zipfile(archive))


if EN_COLAB:
    for nom in ["test.csv", "train.csv"]:
        archive = Path(nom + ".zip")
        if archive.exists() and not zipfile.is_zipfile(archive):
            print(f"{archive} est incomplet ou corrompu : suppression.")
            archive.unlink()

    a_telecharger = [nom for nom in ["test.csv", "train.csv"] if not fichier_valide(nom)]

    if a_telecharger:
        from getpass import getpass

        # Version récente de l'outil Kaggle : elle comprend les nouveaux jetons API
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)

        jeton = getpass("Colle ton jeton API Kaggle (laisse vide pour téléverser kaggle.json à la place) : ").strip()
        if jeton:
            os.environ["KAGGLE_API_TOKEN"] = jeton
        else:
            from google.colab import files

            print("Téléverse ton fichier kaggle.json")
            files.upload()
            dossier_kaggle = Path.home() / ".kaggle"
            dossier_kaggle.mkdir(exist_ok=True)
            Path("kaggle.json").replace(dossier_kaggle / "kaggle.json")
            os.chmod(dossier_kaggle / "kaggle.json", 0o600)

        for nom in a_telecharger:
            subprocess.run(["kaggle", "competitions", "download",
                            "-c", "new-york-city-taxi-fare-prediction", "-f", nom], check=True)

    print("Fichiers prêts :", sorted(p.name for p in Path(".").glob("*.csv*")))

## 1. Charger les données

Fichiers à télécharger : [train.csv et test.csv sur Kaggle](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/data).

Le jeu d'entraînement contient 55 millions de lignes. On n'en lit que les 6 premiers millions. Si tu as `train.csv.zip` (le fichier téléchargé depuis Kaggle), pas besoin de le décompresser : la fonction `lire_csv()` lit directement dans l'archive.

In [ ]:
train = lire_csv("train.csv", nrows=6_000_000)
train.info()

In [ ]:
train.head()

## 2. Variables de date

On convertit `pickup_datetime` en vrai type date, puis on extrait l'année, le mois, le jour, l'heure et le jour de la semaine avec l'accesseur `.dt` (opérations sur toute la colonne).

In [ ]:
def ajouter_dates(df):
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], format="%Y-%m-%d %H:%M:%S UTC")
    df["pickup_date"] = df["pickup_datetime"].dt.date
    df["pickup_day"] = df["pickup_datetime"].dt.day
    df["pickup_hour"] = df["pickup_datetime"].dt.hour
    df["pickup_day_of_week"] = df["pickup_datetime"].dt.day_name()
    df["pickup_month"] = df["pickup_datetime"].dt.month
    df["pickup_year"] = df["pickup_datetime"].dt.year
    return df


train = ajouter_dates(train)
train[["pickup_datetime", "pickup_hour", "pickup_day_of_week", "pickup_month", "pickup_year"]].head()

### Valeurs manquantes

In [ ]:
train.isnull().sum()

## 3. Répartition du montant de la course

In [ ]:
print("Courses avec un tarif négatif :", (train["fare_amount"] < 0).sum())
train = train[train["fare_amount"] >= 0]
print("Courses restantes :", len(train))

Le montant est très asymétrique. On passe au logarithme pour voir sa répartition (on garde les montants strictement positifs : le logarithme de 0 n'est pas défini).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
montants = train.loc[train["fare_amount"] > 0, "fare_amount"]
sns.kdeplot(np.log(echantillon_de(montants, graine=0)), ax=ax)
ax.set_title("Distribution du montant de la course (échelle logarithmique)")
montrer("taxi-montant-log")

## 4. Caractéristiques géographiques

### Coordonnées aberrantes

In [ ]:
coordonnees = ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]
train[coordonnees].agg(["min", "max"])

Des latitudes de -3 488 ou 3 344 n'existent pas, et beaucoup de lignes ont des coordonnées à 0. Pour décider ce qui est « à New York », on regarde l'étendue des coordonnées du jeu de test, qui ne contient pas ces anomalies.

In [ ]:
test = lire_csv("test.csv")
print("Étendue des coordonnées dans le jeu de test (prise en charge et dépose) :")
print("  longitude :", min(test["pickup_longitude"].min(), test["dropoff_longitude"].min()),
      max(test["pickup_longitude"].max(), test["dropoff_longitude"].max()))
print("  latitude  :", min(test["pickup_latitude"].min(), test["dropoff_latitude"].min()),
      max(test["pickup_latitude"].max(), test["dropoff_latitude"].max()))

On fixe les limites utilisées dans l'article (celles de mon analyse d'origine) et on écrit une fonction `dans_zone()` qui travaille sur des colonnes entières : elle sert pour les limites de la ville, les aéroports, les quartiers et le bas de Manhattan.

In [ ]:
def dans_zone(lat, lng, zone):
    """True si le point (lat, lng) est dans le rectangle `zone`."""
    return ((lat >= zone["min_lat"]) & (lat <= zone["max_lat"]) &
            (lng >= zone["min_lng"]) & (lng <= zone["max_lng"]))


limites_ny = {"min_lng": -74.263242, "min_lat": 40.573143,
              "max_lng": -72.986532, "max_lat": 41.709555}

dans_ny = (dans_zone(train["pickup_latitude"], train["pickup_longitude"], limites_ny)
           & dans_zone(train["dropoff_latitude"], train["dropoff_longitude"], limites_ny))

print(f"Courses hors de New York : {(~dans_ny).sum():,}")
train = train[dans_ny]
print(f"Courses restantes : {len(train):,}")

### Lieux de prise en charge et de dépose

Sur New York, en plus de Manhattan, on voit de la densité autour de JFK et de LaGuardia.

In [ ]:
echantillon = echantillon_de(train, graine=0)
limites_ville_lng, limites_ville_lat = (-74.03, -73.75), (40.63, 40.85)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, sens, couleur, titre in [(axes[0], "pickup", "tab:blue", "Prises en charge"),
                                 (axes[1], "dropoff", "tab:green", "Dépôts")]:
    ax.scatter(echantillon[f"{sens}_longitude"], echantillon[f"{sens}_latitude"],
               s=0.5, alpha=0.5, color=couleur)
    ax.set_xlim(limites_ville_lng)
    ax.set_ylim(limites_ville_lat)
    ax.set_title(titre)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
plt.tight_layout()
montrer("taxi-lieux-prise-depose")

## 5. Les aéroports

On définit un rectangle autour de chaque aéroport (EWR est l'aéroport de Newark).

In [ ]:
aeroports = {
    "JFK": {"min_lng": -73.8352, "min_lat": 40.6195, "max_lng": -73.7401, "max_lat": 40.6659},
    "EWR": {"min_lng": -74.1925, "min_lat": 40.6700, "max_lng": -74.1531, "max_lat": 40.7081},
    "la_guardia": {"min_lng": -73.8895, "min_lat": 40.7664, "max_lng": -73.8550, "max_lat": 40.7931},
}

prise_jfk = dans_zone(train["pickup_latitude"], train["pickup_longitude"], aeroports["JFK"])
depose_jfk = dans_zone(train["dropoff_latitude"], train["dropoff_longitude"], aeroports["JFK"])
print("Prises en charge à JFK :", prise_jfk.sum())
print("Dépôts à JFK :", depose_jfk.sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
tous = np.log(echantillon_de(train["fare_amount"].clip(lower=0.01), graine=0))
for ax, masque, titre in [(axes[0], prise_jfk, "Départs de JFK"), (axes[1], depose_jfk, "Arrivées à JFK")]:
    sns.kdeplot(np.log(train.loc[masque, "fare_amount"].clip(lower=0.01)), label=titre, ax=ax)
    sns.kdeplot(tous, label="Toutes les courses", ax=ax)
    ax.set_title(f"Montant (échelle logarithmique) : {titre}")
    ax.legend()
plt.tight_layout()
montrer("taxi-aeroport-jfk")

In [ ]:
def ajouter_aeroports(df):
    for nom, zone in aeroports.items():
        df[f"is_pickup_{nom}"] = dans_zone(df["pickup_latitude"], df["pickup_longitude"], zone).astype(int)
        df[f"is_dropoff_{nom}"] = dans_zone(df["dropoff_latitude"], df["dropoff_longitude"], zone).astype(int)
    return df


train = ajouter_aeroports(train)
train.filter(like="_JFK").mean()

## 6. Distance du trajet (Haversine)

Distance « à vol d'oiseau » en miles entre la prise en charge et la dépose.

In [ ]:
def distance_miles(lat1, lng1, lat2, lng2):
    """Distance de Haversine en miles (accepte des colonnes Pandas entières)."""
    p = np.pi / 180
    a = (0.5 - np.cos((lat2 - lat1) * p) / 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lng2 - lng1) * p)) / 2)
    return 0.6213712 * 12742 * np.arcsin(np.sqrt(a))   # 12742 km = diamètre de la Terre


def ajouter_distance(df):
    df["trip_distance"] = distance_miles(df["pickup_latitude"], df["pickup_longitude"],
                                         df["dropoff_latitude"], df["dropoff_longitude"])
    return df


train = ajouter_distance(train)
train["trip_distance"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
distances = echantillon_de(train["trip_distance"].clip(lower=1e-4), graine=0)
sns.kdeplot(np.log(distances), ax=ax)
ax.set_title("Distribution de la distance du trajet (échelle logarithmique)")
montrer("taxi-distance-log")

Distance et montant : avec les trajets d'aéroport (à gauche), puis sans eux (à droite).

In [ ]:
colonnes_aeroport = [f"is_{sens}_{nom}" for sens in ("pickup", "dropoff") for nom in aeroports]
non_airport = train[train[colonnes_aeroport].sum(axis=1) == 0]
print("Trajets sans aéroport :", len(non_airport))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, donnees, titre in [(axes[0], train, "Tous les trajets"), (axes[1], non_airport, "Sans les aéroports")]:
    ech = echantillon_de(donnees, TAILLE_ECHANTILLON // 2, 0)
    ax.scatter(ech["trip_distance"], ech["fare_amount"], s=4, alpha=0.3)
    ax.set_title(f"Distance vs montant : {titre}")
    ax.set_xlabel("Distance (miles)")
axes[0].set_ylabel("Montant ($)")
plt.tight_layout()
montrer("taxi-distance-vs-montant")

Les trajets de plus de 50 miles facturés très peu sont suspects (une coordonnée erronée, très probablement). Où sont-ils ?

In [ ]:
trajets_longs = non_airport[non_airport["trip_distance"] >= 50]
print("Trajets hors aéroport de plus de 50 miles :", len(trajets_longs))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(trajets_longs["pickup_longitude"], trajets_longs["pickup_latitude"],
           s=8, alpha=0.5, color="tab:blue", label="Prises en charge")
ax.scatter(trajets_longs["dropoff_longitude"], trajets_longs["dropoff_latitude"],
           s=8, alpha=0.5, color="tab:green", label="Dépôts")
ax.set_title("Trajets de plus de 50 miles")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
montrer("taxi-trajets-longs")

## 7. Les quartiers de New York

Des rectangles qui **se chevauchent** : un point qui est dans plusieurs rectangles reçoit le premier nom de la liste. C'est une approximation (pour un résultat exact, utilise les polygones officiels avec GeoPandas).

In [ ]:
quartiers = {
    "manhattan": {"min_lng": -74.0479, "min_lat": 40.6829, "max_lng": -73.9067, "max_lat": 40.8820},
    "queens": {"min_lng": -73.9630, "min_lat": 40.5431, "max_lng": -73.7004, "max_lat": 40.8007},
    "brooklyn": {"min_lng": -74.0421, "min_lat": 40.5707, "max_lng": -73.8334, "max_lat": 40.7395},
    "bronx": {"min_lng": -73.9339, "min_lat": 40.7855, "max_lng": -73.7654, "max_lat": 40.9176},
    "staten_island": {"min_lng": -74.2558, "min_lat": 40.4960, "max_lng": -74.0522, "max_lat": 40.6490},
}


def quartier(lat, lng):
    """Nom du quartier (le premier rectangle qui contient le point)."""
    conditions = [dans_zone(lat, lng, zone) for zone in quartiers.values()]
    return np.select(conditions, list(quartiers.keys()), default="others")


def ajouter_quartiers(df):
    df["pickup_borough"] = quartier(df["pickup_latitude"], df["pickup_longitude"])
    df["dropoff_borough"] = quartier(df["dropoff_latitude"], df["dropoff_longitude"])
    return df


train = ajouter_quartiers(train)
train["pickup_borough"].value_counts()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
echantillon = echantillon_de(train, graine=1)
for ax, nom in zip(axes.ravel(), quartiers):
    for sens, etiquette in [("pickup", "Prise en charge"), ("dropoff", "Dépose")]:
        montants = echantillon.loc[echantillon[f"{sens}_borough"] == nom, "fare_amount"].clip(lower=0.01)
        sns.kdeplot(np.log(montants), label=etiquette, ax=ax)
    ax.set_title(f"Montant (échelle logarithmique) : {nom}")
    ax.legend()
axes.ravel()[-1].axis("off")
plt.tight_layout()
montrer("taxi-montant-quartiers")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for ax, nom in zip(axes.ravel(), quartiers):
    for sens, etiquette in [("pickup", "Prise en charge"), ("dropoff", "Dépose")]:
        distances = echantillon.loc[echantillon[f"{sens}_borough"] == nom, "trip_distance"].clip(lower=1e-4)
        sns.kdeplot(np.log(distances), label=etiquette, ax=ax)
    ax.set_title(f"Distance (échelle logarithmique) : {nom}")
    ax.legend()
axes.ravel()[-1].axis("off")
plt.tight_layout()
montrer("taxi-distance-quartiers")

## 8. Le bas de Manhattan

In [ ]:
bas_manhattan = {"min_lng": -74.0194, "min_lat": 40.6997, "max_lng": -73.9716, "max_lat": 40.7427}


def ajouter_bas_manhattan(df):
    df["is_pickup_lower_manhattan"] = dans_zone(df["pickup_latitude"], df["pickup_longitude"], bas_manhattan).astype(int)
    df["is_dropoff_lower_manhattan"] = dans_zone(df["dropoff_latitude"], df["dropoff_longitude"], bas_manhattan).astype(int)
    return df


train = ajouter_bas_manhattan(train)

manhattan = train[(train["pickup_borough"] == "manhattan") | (train["dropoff_borough"] == "manhattan")]
print("Courses avec au moins une extrémité dans le rectangle de Manhattan :", len(manhattan))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
ech = echantillon_de(manhattan, graine=2)
for ax, sens, titre in [(axes[0], "pickup", "Prise en charge"), (axes[1], "dropoff", "Dépose")]:
    colonne = f"is_{sens}_lower_manhattan"
    for valeur, etiquette in [(1, "Bas de Manhattan"), (0, "Reste de Manhattan")]:
        montants = ech.loc[ech[colonne] == valeur, "fare_amount"].clip(lower=0.01)
        sns.kdeplot(np.log(montants), label=etiquette, ax=ax)
    ax.set_title(f"Montant (échelle logarithmique) : {titre}")
    ax.set_xlabel("Montant (log)")
    ax.legend()
plt.tight_layout()
montrer("taxi-montant-bas-manhattan")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
ech = echantillon_de(manhattan, TAILLE_ECHANTILLON // 2, 3)
for ligne, sens in enumerate(["pickup", "dropoff"]):
    colonne = f"is_{sens}_lower_manhattan"
    for col, (valeur, etiquette) in enumerate([(1, "bas de Manhattan"), (0, "reste de Manhattan")]):
        sous = ech[ech[colonne] == valeur]
        axes[ligne, col].scatter(sous["trip_distance"], sous["fare_amount"], s=4, alpha=0.3)
        axes[ligne, col].set_title(f"{sens} dans le {etiquette}")
for ax in axes[1]:
    ax.set_xlabel("Distance (miles)")
for ax in axes[:, 0]:
    ax.set_ylabel("Montant ($)")
plt.tight_layout()
montrer("taxi-bas-manhattan-dispersion")

## 9. Date et heure de prise en charge

In [ ]:
JOURS = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]


def compter_et_moyenne(df, colonne, ordre=None, valeur="fare_amount", nom=None):
    """Nombre de courses (gauche) et montant moyen (droite) pour chaque valeur de `colonne`."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    nombre = df.groupby(colonne)[valeur].count()
    moyenne = df.groupby(colonne)[valeur].mean()
    if ordre is not None:
        nombre, moyenne = nombre.reindex(ordre), moyenne.reindex(ordre)
    axes[0].bar(nombre.index.astype(str), nombre.values, color="tab:blue")
    axes[0].set_title(f"Nombre de courses selon {colonne}")
    axes[1].bar(moyenne.index.astype(str), moyenne.values, color="tab:orange")
    axes[1].set_title(f"Montant moyen selon {colonne}")
    for ax in axes:
        ax.tick_params(axis="x", labelrotation=45)
    plt.tight_layout()
    montrer(nom or f"taxi-{colonne}")


compter_et_moyenne(train, "pickup_year")

In [ ]:
compter_et_moyenne(train, "pickup_month")

In [ ]:
compter_et_moyenne(train, "pickup_day_of_week", ordre=JOURS)

In [ ]:
compter_et_moyenne(train, "pickup_hour")

Pour les modèles, on encode le jour de la semaine en nombre (dimanche = 0, … samedi = 6), comme dans la version d'origine.

In [ ]:
def encoder_jours(df):
    df["pickup_day_of_week"] = df["pickup_day_of_week"].map({jour: i for i, jour in enumerate(JOURS)})
    return df

## 10. Nombre de passagers

In [ ]:
compter_et_moyenne(train, "passenger_count", nom="taxi-passagers-avant")

In [ ]:
print("Distance moyenne avec 0 passager (miles) :", train.loc[train["passenger_count"] == 0, "trip_distance"].mean())
print("Courses avec plus de 8 passagers :", (train["passenger_count"] > 8).sum())
print("Courses avec plus de 8 passagers dans le jeu de test :", (test["passenger_count"] > 8).sum())

Le jeu de test ne contient aucune course avec plus de 8 passagers : on les supprime du jeu d'entraînement.

In [ ]:
train = train[train["passenger_count"] <= 8]
compter_et_moyenne(train, "passenger_count", nom="taxi-passagers-apres")

## 11. Enregistrer les données nettoyées

In [ ]:
# Variables arrondies (utilisées plus tard dans la partie 2)
def ajouter_coordonnees_arrondies(df):
    for colonne in ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]:
        df[f"{colonne}_round3"] = df[colonne].round(3)
    return df


train = encoder_jours(ajouter_coordonnees_arrondies(train))
train.to_csv("train_cleaned.csv", index=False)
print("train :", train.shape)

### Le jeu de test

Mêmes fonctions, **sans supprimer de lignes** : il faut une prédiction pour chaque course du jeu de test.

In [ ]:
test = lire_csv("test.csv")
test = ajouter_dates(test)
test = ajouter_aeroports(test)
test = ajouter_coordonnees_arrondies(test)
test = ajouter_distance(test)
test = ajouter_quartiers(test)
test = ajouter_bas_manhattan(test)
test = encoder_jours(test)

test.to_csv("test_cleaned.csv", index=False)
print("test :", test.shape)

## 12. Récupérer les graphiques (Google Colab)

Tous les graphiques sont enregistrés en haute résolution dans le dossier `images/`. Dans Colab, cette cellule les regroupe dans une archive et la télécharge sur ton ordinateur.

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    shutil.make_archive("images", "zip", "images")
    files.download("images.zip")
else:
    print("Les graphiques sont dans le dossier :", Path("images").resolve())